# Préparation et nettoyage des données

## 1. Objectif et périmètre

Cette étape consiste à préparer et nettoyer le jeu de données
PhiUSIIL Phishing URL Dataset avant son utilisation dans les
modèles de détection des URL de phishing.

Les décisions de préparation sont fondées sur les observations
réalisées dans le notebook `02_analyse_exploratoire.ipynb`.

### Travail réalisé

- Vérifier la structure et les types des variables.
- Traiter les valeurs manquantes et les valeurs numériques invalides.
- Vérifier les variables binaires et la variable cible.
- Examiner les doublons et les URL répétées.
- Définir le traitement des colonnes et les choix de conservation.
- Exporter le jeu de données préparé et un rapport de nettoyage.

### Limites de cette étape

Cette étape ne réalise pas :
- Le découpage en ensembles d'entraînement et de test.
- L'encodage des variables catégorielles.
- La normalisation ou la standardisation.
- L'entraînement et l'évaluation des modèles.

Ces opérations seront traitées dans les étapes suivantes du projet.

Le fichier original `dataset.csv` est conservé sans modification.

## 2. Présentation des données sources

Le jeu de données utilisé est le PhiUSIIL Phishing URL Dataset.

Le fichier original est stocké dans :

`periode-1/data/raw/dataset.csv`

Il contient initialement :
- 235 795 observations ;
- 56 colonnes ;
- une variable cible nommée `label`.

La variable `label` permet de distinguer les deux catégories :
- `0` : URL de phishing ;
- `1` : URL légitime.

Le fichier original est conservé afin de préserver les données
sources. Les opérations de nettoyage sont réalisées sur une copie.

## 3. Chargement du jeu de données préparé

Après les opérations de nettoyage réalisées avec le script
`src/clean_data.py`, le jeu de données préparé est enregistré
dans `data/processed/dataset_clean.csv`.

Le rapport `data/processed/cleaning_report.json` conserve
les informations relatives aux opérations effectuées.

Dans cette section, nous chargeons les fichiers préparés afin
de vérifier les résultats du nettoyage.

In [5]:
from pathlib import Path
import pandas as pd
import json

# Définition du dossier du projet
PROJECT_DIR = Path.cwd().parent

# Chemins des fichiers préparés
DATA_PATH = PROJECT_DIR / "data" / "processed" / "dataset_clean.csv"
REPORT_PATH = PROJECT_DIR / "data" / "processed" / "cleaning_report.json"

# Vérification des fichiers
print("Dossier utilisé :", PROJECT_DIR)
print("Dataset trouvé :", DATA_PATH.exists())
print("Rapport trouvé :", REPORT_PATH.exists())

# Chargement des données préparées
df_clean = pd.read_csv(DATA_PATH, low_memory=False)

# Chargement du rapport de nettoyage
with open(REPORT_PATH, "r", encoding="utf-8") as file:
    cleaning_report = json.load(file)

print("\nDonnées préparées chargées avec succès.")
print("Dimensions :", df_clean.shape)
print("Valeurs manquantes :", df_clean.isna().sum().sum())

print("\nRépartition de la variable cible :")
print(df_clean["label"].value_counts().sort_index())

Dossier utilisé : c:\Users\ahmat\Documents\Licence_3\projet\Projet_tutoré_ml\projet-ml\periode-1
Dataset trouvé : True
Rapport trouvé : True

Données préparées chargées avec succès.
Dimensions : (235795, 56)
Valeurs manquantes : 32714

Répartition de la variable cible :
label
0    100945
1    134850
Name: count, dtype: int64


## 4. Stratégie de nettoyage des données

Les observations issues de l'analyse exploratoire ont permis
d'identifier plusieurs points nécessitant un traitement avant
l'utilisation du dataset dans les modèles de classification.

Les règles de préparation ont été définies pour corriger les
incohérences tout en préservant les informations utiles.

### 4.1 Gestion des valeurs manquantes et invalides



- Les espaces superflus sont supprimés dans les colonnes textuelles.
- Les colonnes(variables) numériques sont converties vers des types numériques.
- Les valeurs non numériques éventuellement détectées sont contrôlées
  lors de la conversion.
- Les valeurs `Title = 0` sont remplacées par des valeurs manquantes
  lorsque `HasTitle = 0`.

Aucune imputation artificielle n'est réalisée à ce stade.

### 4.2 Gestion des doublons

Aucun doublon complet n'a été détecté dans le dataset.

En revanche, 425 URL distinctes apparaissent plusieurs fois,
représentant 850 lignes concernées.

Ces observations sont conservées dans le dataset préparé.
Leur présence devra être prise en compte lors du découpage
des données afin d'éviter une fuite d'information entre
les ensembles d'entraînement et de test.

### 4.3 Gestion des types de données

Les variables numériques sont converties en types numériques
afin de garantir leur exploitation par les futurs algorithmes.

Les variables textuelles sont conservées.

Les variables binaires et la variable cible sont vérifiées
afin de s'assurer qu'elles respectent les valeurs attendues.

### 4.4 Conservation des observations

Aucune ligne n'est supprimée pendant le nettoyage.

Les valeurs extrêmes ne sont pas supprimées automatiquement :
elles sont conservées afin de ne pas éliminer des informations
potentiellement utiles à la détection du phishing.


## 5. Résultats de l'exécution du nettoyage

Le script `src/clean_data.py` a été exécuté afin d'appliquer
les règles de préparation définies précédemment.

L'exécution s'est terminée sans erreur. Les résultats obtenus
sont enregistrés dans le rapport `cleaning_report.json`.

Les principales observations sont les suivantes :

- Le dataset préparé contient 235 795 lignes et 56 colonnes.
- Aucun doublon complet n'a été détecté.
- 425 URL distinctes apparaissent plusieurs fois, représentant
  850 lignes concernées.
- Aucune ligne n'a été supprimée.
- Les 19 variables binaires ont passé le contrôle des valeurs 0 et 1.
- Les 32 714 valeurs manquantes correspondent aux marqueurs
  d'absence de titre traités lorsque `HasTitle = 0`.
- Aucune valeur numérique invalide n'a été détectée lors de cette
  dernière exécution.

Le fichier original a été conservé et les données préparées
ont été exportées dans `data/processed/dataset_clean.csv`.



## 6. Décisions concernant les variables

À l'issue du nettoyage, les décisions suivantes sont retenues
pour la suite du projet :

- **Variable cible :** la colonne `label` est conservée pour
  la classification binaire. Elle distingue les URL de phishing
  des URL légitimes.
- **Identifiant :** la colonne `FILENAME` est conservée dans
  le fichier préparé, mais ne devra pas être utilisée comme
  variable explicative dans les modèles.
- **Variables textuelles :** les colonnes `URL`, `Domain`,
  `TLD` et `Title` sont conservées sans encodage à ce stade.
- **Variables numériques :** elles sont conservées après
  conversion vers des types numériques.
- **Valeurs manquantes :** aucune imputation n'est effectuée
  pendant cette étape.

Aucune normalisation, standardisation ou séparation des données
n'est réalisée dans cette partie.

Ces opérations seront étudiées dans les étapes ultérieures
du projet, selon le protocole défini par l'équipe.



## 7. Vérification finale du dataset préparé

Après l'exécution du nettoyage, plusieurs contrôles permettent
de vérifier la structure et la cohérence du dataset obtenu.

Les dimensions du fichier préparé sont de 235 795 lignes
et 56 colonnes.

Les contrôles effectués confirment :
- la conservation du nombre initial de lignes ;
- la conservation des 56 colonnes ;
- l'absence de doublons complets ;
- la validité des variables binaires ;
- la validité des valeurs de la variable cible `label`.

Le rapport de nettoyage permet de conserver une trace des
opérations réalisées et de leurs résultats.

Le dataset préparé est ainsi disponible pour les étapes
suivantes du projet.


In [4]:
# Vérification finale du dataset préparé

from pathlib import Path
import pandas as pd
import json

# Récupération du dossier periode-1
PROJECT_DIR = Path.cwd().parent

# Chemins des fichiers
DATA_PATH = PROJECT_DIR / "data" / "processed" / "dataset_clean.csv"
REPORT_PATH = PROJECT_DIR / "data" / "processed" / "cleaning_report.json"

# Chargement du dataset et du rapport
df_clean = pd.read_csv(DATA_PATH, low_memory=False)

with open(REPORT_PATH, "r", encoding="utf-8") as file:
    cleaning_report = json.load(file)

# Récupération des variables binaires depuis le rapport
BINARY_COLUMNS = cleaning_report["binary_columns_checked"]

# Vérification finale
print("Dimensions :", df_clean.shape)

print("\nValeurs manquantes :")
print(df_clean.isna().sum().sum())

print("\nDoublons complets :")
print(df_clean.duplicated().sum())

print("\nValeurs de la variable cible :")
print(sorted(df_clean["label"].dropna().unique()))

print("\nVérification des variables binaires :")

for column in BINARY_COLUMNS:
    values = sorted(df_clean[column].dropna().unique())
    print(f"{column} : {values}")




Dimensions : (235795, 56)

Valeurs manquantes :
32714

Doublons complets :
0

Valeurs de la variable cible :
[np.int64(0), np.int64(1)]

Vérification des variables binaires :
IsDomainIP : [np.int64(0), np.int64(1)]
HasObfuscation : [np.int64(0), np.int64(1)]
IsHTTPS : [np.int64(0), np.int64(1)]
HasTitle : [np.int64(0), np.int64(1)]
HasFavicon : [np.int64(0), np.int64(1)]
Robots : [np.int64(0), np.int64(1)]
IsResponsive : [np.int64(0), np.int64(1)]
HasDescription : [np.int64(0), np.int64(1)]
HasExternalFormSubmit : [np.int64(0), np.int64(1)]
HasSocialNet : [np.int64(0), np.int64(1)]
HasSubmitButton : [np.int64(0), np.int64(1)]
HasHiddenFields : [np.int64(0), np.int64(1)]
HasPasswordField : [np.int64(0), np.int64(1)]
Bank : [np.int64(0), np.int64(1)]
Pay : [np.int64(0), np.int64(1)]
Crypto : [np.int64(0), np.int64(1)]
HasCopyrightInfo : [np.int64(0), np.int64(1)]
NoOfURLRedirect : [np.int64(0), np.int64(1)]
NoOfSelfRedirect : [np.int64(0), np.int64(1)]



Les vérifications finales ont confirmé que les 19 variables
binaires contiennent uniquement les valeurs 0 et 1.
La variable cible `label` respecte également les deux classes
attendues.

Les contrôles de structure et de cohérence du dataset préparé
sont donc satisfaisants.



## 8. Conclusion et transmission des données

Cette étape a permis de préparer le jeu de données destiné
au projet URLShield AI.

Le script `src/clean_data.py` assure le nettoyage et les
vérifications nécessaires, tout en conservant le fichier
original.

Le dataset préparé contient 235 795 lignes et 56 colonnes.
Les contrôles réalisés confirment la cohérence des dimensions,
la validité de la variable cible et des variables binaires,
ainsi que l'absence de doublons complets.

Les fichiers produits sont :

- `data/processed/dataset_clean.csv` : jeu de données préparé.
- `data/processed/cleaning_report.json` : rapport des opérations
  et des contrôles effectués.

Ces fichiers sont mis à disposition de l'équipe pour la suite
du projet.

Le découpage des données et le protocole d'évaluation seront
traités dans la partie dédiée à ces étapes.
